# Hydralab CQR — fine-tuning (weight-update) of trained quantile heads on new synthetic data and automatic detection of NEW LOINCs to train additional HydraLab heads

This notebook (a) reproduces the **initial** CQR head training exactly, then
(b) **fine-tunes every trained head on new data** — warm-starting from the saved weights,
continuing training with a small learning rate, re-fitting the conformal offset, and re-saving in
the identical checkpoint format. This tutorial shows how to incoropriate the extra knowledge (new data) in the synthetic corpus into the heads and show the calibrated **interval scores improve**.

**What maps to the reference notebook**
- Stage 1 embedding: "build_embedding_index" (Clinical ModernBERT, encoder, "pooling='max'",
  question conditioning) 
- Stage 2 and 3 initial heads: "long_to_per_target(include_selection=False)" → "train_all_heads"
  over "DEPTH_LIST=[1,3,5,7]", "alpha=0.05", "hidden=256", "selection=False" — identical.

- "finetune_single_head" / "finetune_all_heads" — warm-start continued training + CQR
  re-calibration, re-saving heads in the same "head_NNNN.pt" + "metadata.json" schema so
  "predict_with_head" / "predict_all_heads" load them unchanged.
- "interval_score" (Winkler; Gneiting & Raftery 2007) + "evaluate_heads_on_holdout".
- "hashing_embedding_index" — an **offline** stand-in for Stage 1 so this notebook is runnable
  without a GPU / HuggingFace download. **Real runs set "EMBEDDING_BACKEND='clinical_modernbert'".**

Note that Splitting is **patient-level** (all of a patient's labs, which share one embedding, stay on one
side) to avoid leaking the shared text across partitions.


In [2]:
#!pip install transformers

NOTE:  IF CPU only, need to uninstall torch to install proper torchvision whl for CPU only devices such that transformers can load HuggingFace models!

In [4]:
# !pip uninstall torch torchaudio torchvision -y
# #!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# !pip install torch torchaudio torchvision --extra-index-url https://download.pytorch.org/whl/cpu

In [5]:
import os, sys, json, pickle, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import torch

#set hydralab CQR module's path to src directory to access python scripts
module_path = os.path.abspath("/home/jupyter/repos/hydralab_cqr/src")
if module_path not in sys.path:
    sys.path.append(module_path)

import module_hydralab_cqr as mh
import module_hydralab_cqr_finetune as ft

In [6]:
# security precaution that prevents attempts at communication with HuggingFace
#if needing ot download weights, set TRANSFORMERS_OFFLINE environment variable to "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1" 

In [7]:
# fine-tuning/updating configuration
SEED = 77030
MODEL_NAME = "Simonlee711/Clinical_ModernBERT" # model to import weights from (e.g., using HuggingFace)
MAX_LENGTH = 512 # token size
BATCH_SIZE = 68  # A100 batches @512 tokens. set to lower for smaller VRAM
ALPHA = 0.05 # 95% conformal PI; quantile levels (0.025, 0.5, 0.975)
HIDDEN = 256 # hidden node size for each layer of fine-tuning
DEPTH_LIST = [1, 3, 5, 7] # hydra head depths; can change this to deeper if there is significant heterogeneity in true data

# parameters for non-GPU updating
EMBEDDING_BACKEND = "clinical_modernbert" #"hashing" # "hashing" is for use without a GPU (note! will be slow)
SURROGATE_NFEATURES = 256 # embedding dim for the offline backend only

TRAIN_INITIAL_HEADS = False #False reuses INITIAL_HEADS_DIRS directory of existing checkpoint weights for each hydralab, True uses INITIAL_HEADS_DIRS
# BASE_FRACTION = 0.35 # proportion of training patients used to train the initial heads
# BASE_BIAS = True # this flag indicates that the user believes the base model under-represents 
# more outlier lab values and should up-sample accordingly; set to False if an unbiased same-distribution split assumed in AoU
MIN_CALIB_SAMPLES = 16 # Conformalized Quantile Regression (CQR) calibration samples
MIN_SAMPLES_HEAD = 32 # initial-head updating samples minimum (at least 16 to fine-tune, 16 to calibrate). 

# fine-tuning parameters
FT_LR = .0001 # warm-start learning rate.
FT_MAX_EPOCHS = 1000
FT_PATIENCE = 10 # stop if loss improvement doesn't occur in 10 epochs
FT_MIN_SAMPLES = 8 # min new obs to fine-tune a head. If not enough, do not update weights
FT_FREEZE_HIDDEN = False # If True, only update the output layer weights. This is useful for very small updating datasets

NEW_HEAD_MINSAMPLE = 32 # min new obs to train a NEW head from scratch

In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# BASE_DIR directory is whichever directory the hydralab heads are stored in
BASE_DIR = Path("/home/jupyter/repos/hydralab_cqr/heads/clinical_modernbert_3/")
# DATA directory is whichever directory the data to fine-tune the hydralab heads are located in
DATA = Path("/home/jupyter/repos/hydralab_cqr/workflow_to_update_hydralab_cqr_example/aou_synthetic_output")
#for new cohort embedding cache (so do not need to rerun embeddings each times--the most expensive compute process in this method)
CACHE_DIR = Path("/home/jupyter/repos/hydralab_cqr/workflow_to_update_hydralab_cqr_example/aou_synthetic_output")

In [9]:
notebook_start_time = datetime.datetime.now()

In [10]:
print("torch", torch.__version__, "and cuda availability:", torch.cuda.is_available())

torch 2.14.1+cpu and cuda availability: False


In [11]:
UPDATE_HEADS_DIR = Path(BASE_DIR / f"heads_depth_3_updated")
print(UPDATE_HEADS_DIR)

/home/jupyter/repos/hydralab_cqr/heads/clinical_modernbert_3/heads_depth_3_updated


In [12]:
(BASE_DIR / "cache").mkdir(parents=True, exist_ok=True)
(UPDATE_HEADS_DIR).mkdir(parents=True, exist_ok=True)

In [13]:
print("device:", DEVICE, "| backend:", EMBEDDING_BACKEND, "| work dir:", BASE_DIR.resolve())

device: cpu | backend: clinical_modernbert | work dir: /home/jupyter/repos/hydralab_cqr/heads/clinical_modernbert_3


### load data (in this case, synthetic, oriented toward metabolic syndrome lab LOINCs)

In [14]:
quant_train_test_labdf = pd.read_csv(DATA / "quant_stack_train_test_labdf.csv", index_col=0)
quant_holdout_labdf    = pd.read_csv(DATA / "quant_holdout_labdf.csv", index_col=0)
print("train:", quant_train_test_labdf.shape, "| holdout:", quant_holdout_labdf.shape)
display(quant_train_test_labdf.head())

code_reference = quant_train_test_labdf[["code", "question"]].drop_duplicates().reset_index(drop=True)
display(code_reference)

train: (2750, 5) | holdout: (704, 5)


,id,code,question,text,value
0,1,2091-7,What is the predicted value for Cholesterol in...,Measurement: Body mass index (BMI) [Ratio] = 4...,33.2
1,1,3043-7,What is the predicted value for Triglyceride [...,Measurement: Body mass index (BMI) [Ratio] = 4...,162.0
2,1,4548-4,What is the predicted value for Hemoglobin A1c...,Measurement: Body mass index (BMI) [Ratio] = 4...,5.7
3,1,1742-6,What is the predicted value for Alanine aminot...,Measurement: Body mass index (BMI) [Ratio] = 4...,37.0
4,1,1920-8,What is the predicted value for Aspartate amin...,Measurement: Body mass index (BMI) [Ratio] = 4...,37.0


,code,question
0,2091-7,What is the predicted value for Cholesterol in...
1,3043-7,What is the predicted value for Triglyceride [...
2,4548-4,What is the predicted value for Hemoglobin A1c...
3,1742-6,What is the predicted value for Alanine aminot...
4,1920-8,What is the predicted value for Aspartate amin...


In [15]:
rng = np.random.default_rng(SEED)
ids = np.array(sorted(quant_train_test_labdf["id"].unique()))
rng.shuffle(ids)
update_ids = set(ids.tolist())

update_df = quant_train_test_labdf[quant_train_test_labdf["id"].isin(update_ids)].reset_index(drop=True)
print(f"UPDATE : {len(update_ids):4d} patients / {len(update_df):5d} rows")

# Show the value-distribution shift the update is meant to fix (mean per code).
shift = update_df.groupby("code")["value"].mean().rename("update_mean")
print("mean value per code (base is biased low when BASE_BIAS=True):")
display(shift.round(2))

UPDATE :  775 patients /  2750 rows
mean value per code (base is biased low when BASE_BIAS=True):


code
1742-6     34.13
1920-8     38.92
2091-7     26.31
3043-7    131.10
4548-4      5.74
Name: update_mean, dtype: float64

## 3. Stage 1 — embeddings

"EMBEDDING_BACKEND='clinical_modernbert'" runs the **exact** reference-notebook embedder. The
offline "'hashing'" backend is a deterministic surrogate (it still carries signal because the
notes contain the phenotype tokens) so the rest of the pipeline can be exercised without a GPU.

In [16]:
help(mh.build_embedding_index)

Help on function build_embedding_index in module module_hydralab_cqr:

build_embedding_index(long_df: 'pd.DataFrame', id_col: 'str', text_col: 'str', model_name: 'str', selection_text_col: 'Optional[str]' = None, question_col: 'Optional[str]' = None, batch_size: 'int' = 16, max_length: 'int' = 512, stride: 'int' = 0, chunk_aggregation: 'str' = 'max', model_kind: 'str' = 'encoder', pooling: 'Optional[str]' = None, trust_remote_code: 'bool' = False, prefix_lm_bidirectional: 'bool' = False, torch_dtype: "Optional[Union[str, 'torch.dtype']]" = None, device: 'Optional[str]' = None) -> 'dict'
    Deduplicate `long_df` by `id_col`, embed each unique text exactly once,
    and (optionally) embed a separate selection-equation text column too.

    This is Stage 1 of the pipeline. The output is consumed by
    `long_to_per_target` to build per-head training data.

    Args:
        long_df:             Long-form DataFrame; one row per
                             (id, code, value) observation. T

In [17]:
embed_start_time = datetime.datetime.now()

In [18]:
%%time
##Note:  this is the most expensive stage. run once to save embeddings and id orders for reference in all downstream hydra heads. 
update_result = mh.build_embedding_index(
    update_df,
    id_col='id',
    text_col='text',
    question_col='question',     # set to None to skip question conditioning
    model_name=MODEL_NAME,
    model_kind= 'encoder', #'generative', 
    prefix_lm_bidirectional=True, #change to True if a Hierarchical Reasoning Model
    pooling= 'max', #'last',
    # selection_text_col=None  (default) - no separate selection text
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    stride=0,                    # >0 for overlapping chunks on very long text
    chunk_aggregation='max', # or 'mean'
    device=DEVICE
)

[warn] 741 ids have multiple distinct questions; using the first per id.
[embed] outcome (encoder): 2750 long-form rows -> 775 unique ids to embed WITH question conditioning


Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Simonlee711/Clinical_ModernBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Embedding chunks:   0%|          | 0/14 [00:00<?, ?it/s]

[embed_q] done. shape=(775, 768)
CPU times: user 27min 27s, sys: 4min 43s, total: 32min 11s
Wall time: 16min 22s


In [19]:
%%time
##Note:  this is the most expensive stage. run once to save embeddings and id orders for reference in all downstream hydra heads. 
holdout_result = mh.build_embedding_index(
    quant_holdout_labdf,
    id_col='id',
    text_col='text',
    question_col='question',     # set to None to skip question conditioning
    model_name=MODEL_NAME,
    model_kind= 'encoder', #'generative', 
    prefix_lm_bidirectional=True, #change to True if a Hierarchical Reasoning Model
    pooling= 'max', #'last',
    # selection_text_col=None  (default) - no separate selection text
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    stride=0,                    # >0 for overlapping chunks on very long text
    chunk_aggregation='max', # or 'mean'
    device=DEVICE
)

[warn] 188 ids have multiple distinct questions; using the first per id.
[embed] outcome (encoder): 704 long-form rows -> 194 unique ids to embed WITH question conditioning


Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: Simonlee711/Clinical_ModernBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Embedding chunks:   0%|          | 0/4 [00:00<?, ?it/s]

[embed_q] done. shape=(194, 768)
CPU times: user 6min 59s, sys: 1min 13s, total: 8min 13s
Wall time: 4min 11s


In [20]:
embed_end_time = datetime.datetime.now()
print(embed_end_time)
embed_run_time_minutes = np.round((embed_end_time - embed_start_time).total_seconds()/60, 2)
print(f"embedding run time = {embed_run_time_minutes} minutes")

2026-10-07 20:53:08.920250
embedding run time = 20.65 minutes


In [21]:
update_embeddings = update_result['embeddings']
update_id_to_row = update_result['id_to_row']
update_id_order = update_result['id_order']

print(f'update embeddings shape: {update_embeddings.shape}')

holdout_embeddings = holdout_result['embeddings']
holdout_id_to_row = holdout_result['id_to_row']
holdout_id_order = holdout_result['id_order']

print(f'holdout embeddings shape: {holdout_embeddings.shape}')

update embeddings shape: (775, 768)
holdout embeddings shape: (194, 768)


In [22]:
# Cache for re-runs if needed
np.save(CACHE_DIR / "update_embeddings.npy", update_embeddings)
Path(CACHE_DIR / "update_id_to_row.json").write_text(json.dumps(update_id_to_row))
with open(CACHE_DIR / "update_id_to_row.pkl", "wb") as f:
    pickle.dump(update_id_to_row, f)
Path(CACHE_DIR / "update_id_order.json").write_text(json.dumps(update_id_order, default=str))
print(f'Shape: {update_embeddings.shape}')

Shape: (775, 768)


In [23]:
# Cache for re-runs if needed
np.save(CACHE_DIR / "holdout_embeddings.npy", holdout_embeddings)
Path(CACHE_DIR / "holdout_id_to_row.json").write_text(json.dumps(holdout_id_to_row))
with open(CACHE_DIR / "holdout_id_to_row.pkl", "wb") as f:
    pickle.dump(holdout_id_to_row, f)
Path(CACHE_DIR / "holdout_id_order.json").write_text(json.dumps(holdout_id_order, default=str))
print(f'Shape: {holdout_embeddings.shape}')

Shape: (194, 768)


## reload cached embeddings if notebook shutdown and want to avoid re-embedding (time consuming on CPU only devices)

In [24]:
# Cache for re-runs if needed (can comment out code blocks above that do the embedding)
update_embeddings = np.load(CACHE_DIR / "update_embeddings.npy")
with open(CACHE_DIR / "update_id_to_row.pkl", "rb") as f:
    update_id_to_row = pickle.load(f)
with open(CACHE_DIR / "update_id_order.json", "r") as file:
    update_id_order = json.load(file)

In [25]:
# Cache for re-runs if needed (can comment out code blocks above that do the embedding)
holdout_embeddings = np.load(CACHE_DIR / "holdout_embeddings.npy")
with open(CACHE_DIR / "holdout_id_to_row.pkl", "rb") as f:
    holdout_id_to_row = pickle.load(f)
with open(CACHE_DIR / "holdout_id_order.json", "r") as file:
    holdout_id_order = json.load(file)

## 5. Update — fine-tune every head on the new (UPDATE) data
"finetune_all_heads" warm-starts each head from its saved weights, continues training on the new
data at a low learning rate (keeping the head's stored "y_mean/y_std" fixed so the pre-trained
weights reflect AoU reference checkpoints), re-fits the CQR offset "Q_hat" on a fresh calibration split, and re-saves in
the identical schema. Heads with too few new observations are not updated

In [26]:
%%time
update_meta = ft.finetune_all_heads(
    initial_heads_dir=BASE_DIR, 
    updated_heads_dir=UPDATE_HEADS_DIR,
    new_long_df=update_df, 
    new_embeddings=update_embeddings, 
    new_id_to_row=update_id_to_row,
    id_col="id", 
    code_col="code", 
    value_col="value",
    alpha=ALPHA, 
    val_size=0.15, 
    calib_size=0.15,
    min_calib_samples=MIN_CALIB_SAMPLES, 
    min_samples_per_head=FT_MIN_SAMPLES,
    lr=FT_LR, 
    max_epochs=FT_MAX_EPOCHS, 
    base_patience=FT_PATIENCE,
    freeze_hidden=FT_FREEZE_HIDDEN, 
    recalibrate=True,
    train_new_heads=True,
    min_samples_new_head=NEW_HEAD_MINSAMPLE,
    random_state=SEED, 
    device=DEVICE)

[long->targets] 5 unique codes ready for training 
CPU times: user 5.96 s, sys: 1.13 s, total: 7.1 s
Wall time: 4.67 s


In [27]:
diag = pd.DataFrame([e for e in update_meta if e.get("finetuned")])
keep_cols = ["name", "n_finetune_obs", "epochs_run_finetune",
        "val_pinball_before", "val_pinball_after", "Q_hat_before", "Q_hat_after"]
print(f"Fine-tuning diagnostics:")
display(diag[keep_cols].round(4))

Fine-tuning diagnostics:


,name,n_finetune_obs,epochs_run_finetune,val_pinball_before,val_pinball_after,Q_hat_before,Q_hat_after
0,2091-7,622,29,0.1637,0.1583,-0.1551,14.0872
1,3043-7,622,22,0.1717,0.1647,1.8621,67.5168
2,4548-4,494,42,0.0946,0.0742,0.0281,0.0871


### 5b. Head-set growth — the newly trained heads on new LOINCS previously not in heads trained, but in new datasets

Each updated directory now contains the fine-tuned original heads **plus** a new head per unseen
code, in the same `head_NNNN.pt` + `metadata.json` format `predict_all_heads` loads.

In [28]:
im = json.loads((BASE_DIR / "metadata.json").read_text())
um = json.loads((UPDATE_HEADS_DIR / "metadata.json").read_text())
n_init = sum(e.get("status") == "trained" for e in im)
n_upd  = sum(e.get("status") == "trained" for e in um)
new_entries = [e for e in um if e.get("newly_trained") and e.get("status") == "trained"]
grown_df = pd.DataFrame(dict(initial_heads=n_init, 
                      updated_heads=n_upd,
                      new_heads=len(new_entries),
                      new_codes=", ".join(e["name"] for e in new_entries)), index=[0])
print("Head-set growth per depth:")
display(grown_df)

Head-set growth per depth:


,initial_heads,updated_heads,new_heads,new_codes
0,500,502,2,"1742-6, 1920-8"


In [29]:
# Show the new-head metadata for one depth (idx continues after the originals).
newmeta = [e for e in json.loads((UPDATE_HEADS_DIR / "metadata.json").read_text())
           if e.get("newly_trained")]
# show = ["idx", "name", "status", "newly_trained", "n_new_obs", "outcome_depth",
#         "calibration_status", "n_train", "n_val"]
print(f"New-head metadata (depth 3):")
newmeta_df = pd.DataFrame(newmeta)
display(newmeta_df) #[[c for c in show if any(c in e for e in newmeta)]])

New-head metadata (depth 3):


,idx,name,status,selection_used,outcome_depth,epochs_run_quantile,best_val_pinball_standardized,y_train_mean,y_train_std,y_train_min,...,Q_hat,alpha,target_coverage,quantile_level_used,empirical_coverage_calib,mean_interval_half_width_calib,raw_coverage_pre_cqr,raw_interval_half_width_pre_cqr,Q_hat_sign,n_calib
0,504,1742-6,trained,False,3,27,0.184249,34.016949,12.821008,9.0,...,1.960243,0.05,0.95,0.973684,0.986842,31.527039,0.947368,29.566788,positive (interval expanded),76
1,505,1920-8,trained,False,3,15,0.200631,37.720339,19.976774,10.0,...,-0.887060,0.05,0.95,0.973684,0.986842,44.809277,0.986842,45.696339,negative (interval shrunk),76


In [30]:
# Confirm the update actually changed the weights of every fine-tuned head.
print("mean abs(delta between weights) of first Linear layer between initial weights (base model) and updated:")
meta = json.loads((BASE_DIR / "metadata.json").read_text())
deltas = []
for e in meta:
    if e.get("status") != "trained":
        continue
    a = torch.load(BASE_DIR / f"head_{e['idx']:04d}.pt", weights_only=False, map_location=DEVICE)
    b = torch.load(UPDATE_HEADS_DIR / f"head_{e['idx']:04d}.pt", weights_only=False, map_location=DEVICE)
    wa = a["quantile_state_dict"]["net.0.weight"]; wb = b["quantile_state_dict"]["net.0.weight"]
    deltas.append((e["name"], float((wb - wa).abs().mean())))
delta_df = pd.DataFrame(deltas, columns=["code", "delta_net_0_weight"]).sort_values(by=["delta_net_0_weight"], ascending=False)
delta_df.loc[delta_df["delta_net_0_weight"]!=0,]

mean abs(delta between weights) of first Linear layer between initial weights (base model) and updated:


,code,delta_net_0_weight
166,2091-7,0.002382
371,4548-4,0.002137
271,3043-7,0.001537


## 6. Evaluate INITIAL vs UPDATED on the holdout

Winkler **interval score** (lower is better — rewards narrow intervals that still cover),
empirical **coverage** (target = 1 − α = 0.95), and mean interval **width**.

In [31]:
rows = []
percode_updated = {}
ev_i = ft.evaluate_heads_on_holdout(BASE_DIR, 
                                    quant_train_test_labdf,
                                    update_embeddings,
                                    update_id_order,
                                    alpha=ALPHA)
ev_u = ft.evaluate_heads_on_holdout(UPDATE_HEADS_DIR, 
                                    quant_holdout_labdf,
                                    holdout_embeddings,
                                    holdout_id_order,
                                    alpha=ALPHA)
# Fair before/after updated fine-tuning: restrict to the ORIGINAL codes that both stages share.
ev_i_o = ev_i[ev_i["code"].isin(code_reference["code"])]
ev_u_o = ev_u[ev_u["code"].isin(code_reference["code"])]
for stage, ev in [("initial", ev_i_o), ("updated", ev_u_o)]:
    rows.append(dict(depth = 3,
                     stage=stage, n=len(ev),
                     interval_score=ev["interval_score"].mean(),
                     coverage=ev["covered"].mean(), width=ev["width"].mean()))
wis_holdout_comp_summary = pd.DataFrame(rows).round(3)
print("Original shared codes only (fair initial vs updated comparison):")
display(wis_holdout_comp_summary)

Applying heads:   0%|          | 0/504 [00:00<?, ?it/s]

Applying heads:   0%|          | 0/506 [00:00<?, ?it/s]

Original shared codes only (fair initial vs updated comparison):


,depth,stage,n,interval_score,coverage,width
0,3,initial,1738,147.493,0.970,92.358002
1,3,updated,704,139.739,0.977,119.433998


In [32]:
piv_wis = wis_holdout_comp_summary.pivot(index="depth", columns="stage", values="interval_score")
piv_wis["improvement_%"] = (piv_wis["initial"] - piv_wis["updated"]) / piv_wis["initial"] * 100.0
piv_wis = piv_wis.round(3)
print("Interval-score improvement on original codes from initial base trained on AoU and then updated updated), higher % = better:")
display(piv_wis)

Interval-score improvement on original codes from initial base trained on AoU and then updated updated), higher % = better:


stage,initial,updated,improvement_%
depth,,,
3,147.493,139.739,5.257


### 6b. Holdout performance of the NEW heads

The new heads didn't exist before the update. Here is their holdout coverage / width / interval
score (one depth shown), alongside the original codes for context.

In [33]:
pc = ev_u.groupby("code").agg(n=("value", "size"), coverage=("covered", "mean"),
                              width=("width", "mean"),
                              interval_score=("interval_score", "mean")).round(3).reset_index()
pc["head_type"] = np.where(pc.index.isin(code_reference["code"]), "NEW (trained in update)", "original (fine-tuned)")
print(f"Per-code holdout eval on UPDATED heads:")
pc.sort_values("interval_score", inplace=True)
display(pc)

Per-code holdout eval on UPDATED heads:


,code,n,coverage,width,interval_score,head_type
4,4548-4,120,0.958,1.624000,1.938,original (fine-tuned)
0,1742-6,126,0.984,63.116001,68.168,original (fine-tuned)
2,2091-7,166,0.982,71.003998,83.773,original (fine-tuned)
1,1920-8,126,0.976,89.757004,90.973,original (fine-tuned)
3,3043-7,166,0.982,318.303009,386.661,original (fine-tuned)


## 7. Save evaluation and updated heads

In [34]:
wis_holdout_comp_summary_path = UPDATE_HEADS_DIR / "initial_vs_updated_holdout_eval.csv"
wis_holdout_comp_summary.to_csv(wis_holdout_comp_summary_path, index=False)
print("Wrote:", wis_holdout_comp_summary_path)
print("\nUpdated head directories (each has head_NNNN.pt + metadata.json, loadable by predict_all_heads):")
n_pt = len(list(UPDATE_HEADS_DIR.glob("head_*.pt")))
print(f" {UPDATE_HEADS_DIR}  ({n_pt} heads)")

Wrote: /home/jupyter/repos/hydralab_cqr/heads/clinical_modernbert_3/heads_depth_3_updated/initial_vs_updated_holdout_eval.csv

Updated head directories (each has head_NNNN.pt + metadata.json, loadable by predict_all_heads):
 /home/jupyter/repos/hydralab_cqr/heads/clinical_modernbert_3/heads_depth_3_updated  (502 heads)


In [35]:
notebook_end_time = datetime.datetime.now()
notebook_run_time_minutes = np.round((notebook_end_time - notebook_start_time).total_seconds()/60, 2)
print(f"notebook run time = {notebook_run_time_minutes} minutes")

notebook run time = 22.13 minutes
